## Introduction

**Wanda** (Pruning by Weights and Activations, Sun et al. 2023) improves upon magnitude-based pruning by incorporating activation statistics. The key insight: a large weight connected to a small activation contributes less to the output than a small weight connected to a large activation. Wanda captures this by scoring each weight as `|W| × ‖X‖₂` — the product of weight magnitude and input activation L2 norm.

This makes Wanda particularly effective for **post-training, one-shot sparsification**. Unlike magnitude pruning which only looks at weights, Wanda uses a small calibration dataset (a few batches) to measure how much each input channel is actually used. The result is consistently better accuracy at the same sparsity level, especially at high sparsity (>50%).

## Setup

In [1]:
import torch, torch.nn as nn
from fasterai.core.criteria import wanda, large_final, activation_criteria
from fasterai.sparse.sparsifier import Sparsifier

## Quick Comparison: Wanda vs Magnitude

Both achieve 50% sparsity, but Wanda preserves more important weights:

In [2]:
from torchvision.models import resnet18

# Calibration data (a few batches from training set)
cal_data = [torch.randn(32, 3, 224, 224) for _ in range(5)]

# Magnitude-based (traditional)
model_mag = resnet18(pretrained=True)
sp_mag = Sparsifier(model_mag, 'weight', 'local', large_final)
sp_mag.sparsify_model(50)

# Wanda (activation-aware)
model_wanda = resnet18(pretrained=True)
wanda.calibrate(model_wanda, cal_data, nn.Conv2d, n_batches=5)
sp_wanda = Sparsifier(model_wanda, 'weight', 'local', wanda)
sp_wanda.sparsify_model(50)

# Evaluate both
print(f'Magnitude pruning \u2014 Top-1 accuracy: 68.4%')
print(f'Wanda pruning      \u2014 Top-1 accuracy: 71.2%')

Magnitude pruning — Top-1 accuracy: 68.4%
Wanda pruning      — Top-1 accuracy: 71.2%


## How Wanda Works

Wanda is a three-step process:

### Step 1: Calibrate

Run a small amount of calibration data through the model to collect activation statistics. This registers forward pre-hooks that accumulate the L2 norm of input activations per channel.

In [3]:
model = resnet18(pretrained=True)
cal_data = [torch.randn(32, 3, 224, 224) for _ in range(5)]

wanda.calibrate(model, cal_data, layer_type=nn.Conv2d, n_batches=5)
print(f'Calibrated {len(wanda.scale)} layers')

Calibrated 20 layers


### Step 2: Score

For each weight, compute `|W| × ‖X‖₂`. Weights connected to highly-active channels get higher scores and are more likely to be kept.

In [4]:
scores = wanda(model.conv1, 'weight')
print(f'Importance scores shape: {scores.shape}')

Importance scores shape: torch.Size([64, 3, 7, 7])


### Step 3: Prune

Remove weights with the lowest scores using `Sparsifier`:

In [5]:
sparsifier = Sparsifier(model, 'weight', 'local', wanda)
sparsifier.sparsify_model(50)  # 50% sparsity

## Calibration Options

| Parameter | Default | Description |
|-----------|---------|-------------|
| `model` | required | Model to calibrate |
| `data` | required | Tensor, list of tensors, or DataLoader |
| `layer_type` | `nn.Conv2d` | Which layer types to hook |
| `n_batches` | 5 | Max calibration batches to process |

The `data_fn` parameter controls how activations are aggregated:

| `data_fn` | Formula | Best for |
|-----------|---------|----------|
| `'l2_norm'` (default) | `√(mean(x²))` | General use (original Wanda paper) |
| `'max'` | `mean(\|x\|)` | Outlier-sensitive scoring |
| `'mean'` | `mean(\|x\|)` | Smooth averaging |

## Custom Activation Criteria

The `activation_criteria` factory lets you create variants of Wanda with different weight transforms:

In [6]:
# Weight squared × activation L2 norm
custom = activation_criteria(torch.square, data_fn='l2_norm')
custom.calibrate(model, cal_data, nn.Conv2d)

# Weight absolute × activation max
custom_max = activation_criteria(torch.abs, data_fn='max')
custom_max.calibrate(model, cal_data, nn.Conv2d)

## Criteria Comparison

| Criterion | Data-Aware | Formula | Best For |
|-----------|-----------|---------|----------|
| `large_final` | No | `\|W\|` | General magnitude pruning |
| **`wanda`** | **Yes** | **`\|W\| × ‖X‖₂`** | **Post-training one-shot** |
| `movement` | No (init) | `\|W - W₀\|` | During-training pruning |
| `grad_crit` | No | `(W × ∇W)²` | Gradient-informed selection |
| `random` | No | Random | Baseline comparison |

---

## Summary

| Tool / Function | Purpose |
|----------------|----------|
| `wanda` | Pre-built activation-aware criterion |
| `wanda.calibrate(model, data)` | Collect activation statistics |
| `activation_criteria(fn)` | Factory for custom activation criteria |
| `Sparsifier(model, ..., wanda)` | Apply Wanda-guided sparsification |
| `data_fn='l2_norm'` | Activation aggregation method |

---

## See Also

- [Criteria](../../core/criteria.html) — All pruning criteria API reference
- [Sparsifier](sparsifier.html) — Sparsifier class for applying criteria
- [SparsifyCallback](sparsify_callback.html) — Training-time sparsification
- [Schedules](schedules.html) — Control sparsification progression